# Kafka + PySpark Streaming Walkthrough

A complete guide to building real-time data pipelines using Apache Kafka for ingestion and PySpark Streaming for processing.

**Notebook Structure:**
1. Setup instructions (Kafka/ZooKeeper on Windows)
2. Conceptual foundations (Slides 1-6)
3. PySpark implementation (Slides 7-9)
4. Real-world applications (Slides 10-11)

## SETUP: Windows Installation and Configuration

### Prerequisites
- Java 11+ installed and on PATH (verify with: `java -version`)
- Apache Kafka downloaded from https://kafka.apache.org/downloads
- Extract Kafka to a folder (e.g., C:\Kafka)

### Terminal 1: Start ZooKeeper

Open Command Prompt (cmd) as Administrator and run:

```cmd
cd C:\Kafka
.\bin\windows\zookeeper-server-start.bat .\config\zookeeper.properties
```

Wait for this output:
```
[2024-06-02 10:15:30,123] INFO Binding to port 2181
[2024-06-02 10:15:30,456] INFO binding to port 0.0.0.0/0.0.0.0:2181
[2024-06-02 10:15:30,789] INFO Server startup completed
```

**Keep this terminal open.**

### Terminal 2: Start Kafka Broker

Open a NEW Command Prompt window and run:

```cmd
cd C:\Kafka
.\bin\windows\kafka-server-start.bat .\config\server.properties
```

Wait for this output:
```
[2024-06-02 10:16:45,789] INFO Broker 0 started
[2024-06-02 10:16:45,890] INFO KafkaServer id = 0
[2024-06-02 10:16:46,123] INFO Fetcher started
```

**Keep this terminal open.**

### Terminal 3: Create Topic

Open a NEW Command Prompt window and run:

```cmd
cd C:\Kafka
.\bin\windows\kafka-topics.bat --create --topic device-data --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1
```

Expected output:
```
Created topic device-data.
```

Verify topic creation:
```cmd
.\bin\windows\kafka-topics.bat --list --bootstrap-server localhost:9092
```

Should show:
```
device-data
```

### Terminal 4: Send Test Data

Open a NEW Command Prompt window and run:

```cmd
cd C:\Kafka
.\bin\windows\kafka-console-producer.bat --broker-list localhost:9092 --topic device-data
```

You'll see: `>`

Paste these JSON messages one per line (right-click to paste):

```json
{"payload": {"device_id": "A1", "metrics": {"temp": 42}}}
{"payload": {"device_id": "B2", "metrics": {"temp": 38}}}
{"payload": {"device_id": "C3", "metrics": {"temp": 45}}}
{"payload": {"device_id": "A1", "metrics": {"temp": 43}}}
{"payload": {"device_id": "B2", "metrics": {"temp": 39}}}
```

Press Ctrl+C to stop. **Keep this terminal available for sending more data as needed.**

### Terminal Summary

You should now have these running:

| Terminal | Command | Status | Keep Open |
|---|---|---|---|
| 1 | zookeeper-server-start.bat | Active (coordination) | YES |
| 2 | kafka-server-start.bat | Active (broker at :9092) | YES |
| 3 | kafka-topics.bat | Used once (create topic) | Optional |
| 4 | kafka-console-producer.bat | Active (send messages) | YES |
| This | Jupyter Notebook | PySpark Consumer | YES |

**Now proceed to the code cells below.**

## Conceptual Foundation: Kafka and Spark

### 1. Why Kafka Needs a Processing Engine

| Role | Tool | Strength |
|---|---|---|
| **Transport** | Apache **Kafka** | Durable, ordered, replayable event ingestion |
| **Factory** | Apache **Spark** | Aggregations, SQL joins, ML on data in-flight |

**The missing piece:** Kafka excels at routing and retaining raw data but does not perform complex computations. Aggregations, SQL joins, and stream filtering all need a dedicated engine like Spark.

**Key insight:** Kafka is the pipe, Spark is the treatment plant. You need both for a complete system.

### 2. Apache Kafka - The Decoupling Event Broker

**The problem:** Without Kafka, the Order Service must know every consumer's address, retry on failure, and slow down to the speed of its slowest dependency.

**The fix:** With Kafka, drop one event into a topic and walk away. Each consumer reads at its own pace. Add a 6th consumer with zero upstream changes.

**Four core roles:**

| Role | Responsibility |
|---|---|
| **Producer** | Writes events to Kafka |
| **Broker** | Server that stores the log on disk |
| **Topic** | Named append-only log |
| **Consumer** | Pulls events at its own pace |

**Architecture:**
```
Producer -> [Topic: orders] -> Consumer 1 (billing)
                           -> Consumer 2 (shipping)
                           -> Consumer 3 (analytics)
```

### 3. Topics and Partitions - Horizontal Scaling

**Topics:** Named logs for one event type
- `clicks` = page views
- `orders` = checkouts
- `device-data` = IoT readings

**Partitions:** Each topic is split into N ordered lanes
```
Topic: device-data
  Partition 0: [event1, event2, event3, ...]
  Partition 1: [event4, event5, event6, ...]
  Partition 2: [event7, event8, event9, ...]
```

**Two critical rules:**
1. Same key = same partition (ordering guaranteed for that key)
2. Different partitions can be read in parallel (horizontal scale)

### 4. Consumer Groups - Load Distribution

**Problem:** One consumer gets buried with millions of events per second.

**Solution:** Spin up multiple consumers with the same `group_id`. Kafka load-balances.

```
Topic: device-data (3 partitions)
  Consumer Group: analytics
    Consumer 1 reads Partition 0
    Consumer 2 reads Partition 1
    Consumer 3 reads Partition 2
```

**Three guarantees:**
1. Load-balanced: each partition assigned to exactly one consumer
2. No duplicates: same partition never read by two consumers simultaneously
3. Fault-tolerant: if Consumer 1 crashes, Partition 0 reassigned within seconds

### 5. Kafka vs Traditional Message Queues

| Feature | Message Queue | Kafka |
|---|---|---|
| Storage | Deleted on read | Persistent immutable log |
| Lifetime | Temporary pipe | Configurable retention |
| Consumer crash | Data lost forever | Resumes from last offset |
| Late arrival | Only sees future events | Can replay from offset 0 |
| Multi-subscriber | One drains it | N independent groups |

**Bottom line:** Traditional queues are pipes that destroy data. Kafka is a durable log that retains everything.

### 6. The 4-Stage Big Data Ecosystem

| Stage | Purpose | Tools |
|---|---|---|
| **1. Ingestion** | Capture millions of real-time events | Kafka, Kinesis, Pub/Sub |
| **2. Processing** | Clean, transform, run ML models | Spark Streaming, Flink |
| **3. Storage** | Save refined data durably | Delta/Iceberg, Snowflake, BigQuery |
| **4. Visualization** | Live dashboards for business | Tableau, Power BI, Grafana |

**Complete pipeline:**
```
Microservices -> Kafka -> Spark Streaming -> Database -> Dashboard
   (raw)        (buffer)    (processing)     (facts)    (insight)
```

## PySpark Implementation Section

Now we'll write Python code to:
1. Read from Kafka topic
2. Parse JSON payloads
3. Write output to console

**Before running code cells:**
- Ensure all 4 terminals from Setup are still running
- Terminal 4 should have sent test data to Kafka
- If you see errors, check the terminal logs for issues

### 7. PySpark - Reading the Stream

**Kafka DataFrame columns:**

| Column | Type | Content |
|---|---|---|
| `key` | BINARY | Optional message key |
| `value` | BINARY | Message payload (JSON as bytes) |
| `topic` | STRING | Source topic name |
| `partition` | INTEGER | Partition number |
| `offset` | LONG | Position in partition |
| `timestamp` | TIMESTAMP | When message arrived |
| `timestampType` | INTEGER | Type (0=CreateTime, 1=LogAppendTime) |

**Connection options:**

| Option | Meaning | Value |
|---|---|---|
| `kafka.bootstrap.servers` | Broker address | localhost:9092 |
| `subscribe` | Topic to read | device-data |
| `startingOffsets` | Start from | earliest (or latest) |

**Important:** The `value` column is binary bytes. Must cast to STRING before parsing JSON.

In [ ]:
from pyspark.sql import SparkSession

# Initialize Spark Session with Kafka connector
spark = (
    SparkSession.builder
        .appName("kafka-pyspark-demo")
        .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.0.0")
        .getOrCreate()
)

print("Spark Session created successfully")

# Read Kafka stream
kafka_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'localhost:9092') \
    .option('subscribe', 'device-data') \
    .option('startingOffsets', 'earliest') \
    .load()

print("Connected to Kafka broker")
print("\nKafka DataFrame Schema:")
kafka_df.printSchema()

### 8. PySpark - Parsing JSON Payload

**Why parsing is needed:**

Kafka gives you raw bytes. We need to:
1. Cast bytes to string
2. Parse JSON structure
3. Extract fields

**Transformation:**
```
Input:  BINARY  -> {"payload": {"device_id": "A1", "metrics": {"temp": 42}}}
Step 1: STRING  -> {"payload": {"device_id": "A1", "metrics": {"temp": 42}}}
Step 2: PARSED  -> value_json: {payload: {device_id: A1, metrics: {temp: 42}}}
Step 3: FLAT    -> payload: {device_id: A1, metrics: {temp: 42}}
```

**Output we want:**
| device_id | metrics |
|---|---|
| A1 | {"temp": 42} |

In [ ]:
from pyspark.sql.functions import col, from_json, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Step 1: Define schema matching our Kafka messages
# Message format: {"payload": {"device_id": "A1", "metrics": {"temp": 42}}}

json_schema = StructType([
    StructField("payload", StructType([
        StructField("device_id", StringType()),
        StructField("metrics", StructType([
            StructField("temp", IntegerType()),
        ])),
    ])),
])

print("Schema defined")

# Step 2: Cast binary value to string
kafka_json_df = kafka_df.withColumn('value', expr('CAST(value AS STRING)'))
print("Step 2: Binary cast to STRING")

# Step 3: Parse JSON using schema
streaming_df = kafka_json_df.withColumn('value_json', from_json(col('value'), json_schema))
print("Step 3: JSON parsed with schema")

# Step 4: Flatten nested structure
flattened_df = streaming_df.select(expr('value_json.*'))
print("Step 4: Flattened to top-level columns")

print("\nFinal DataFrame Schema:")
flattened_df.printSchema()

### 9. PySpark - Writing the Stream to Console

**Three critical knobs:**

| Knob | What it does | Options |
|---|---|---|
| **Output Mode** | What rows to write | append, update, complete |
| **Checkpointing** | Track offset on disk | Directory path |
| **Sink** | Where to write | console, parquet, memory, Kafka |

**Output modes:**
- `append`: Only new rows (for unbounded streams)
- `update`: Only changed rows since last checkpoint
- `complete`: Entire result table (only with aggregation)

**Checkpoint:** Saves the exact offset Spark has read. On restart, resumes from that point.

**Console output example:**
```
-------------------------------------------
Batch: 0
-------------------------------------------
+--------+----------+
|device_id|metrics   |
+--------+----------+
|A1       |{temp: 42}|
|B2       |{temp: 38}|
+--------+----------+
```

In [ ]:
# Write stream to console
# This will continuously print batches of new events

query = flattened_df.writeStream \
    .format('console') \
    .outputMode('append') \
    .option('checkpointLocation', './checkpoint_kafka_demo') \
    .option('numRows', 100) \
    .start()

print("Stream started. Waiting for events...")
print("This cell will block until you stop it (Ctrl+C in notebook)")
print()

# Keep consuming events
query.awaitTermination()

### 10. Real-World Applications

**Security - Fraud Detection:**
```
Transaction Event -> Kafka -> Spark ML Model -> Alert if fraud > 0.9
```
Detects bot attacks and credit card fraud with 90%+ accuracy in real time.

**Analytics - Live Dashboards:**
```
Sensor Events -> Kafka -> Spark Aggregations -> Dashboard (refresh every 5s)
```
Vehicle locations, retail sales, stock prices updated live for executives.

**E-Commerce - Recommendations:**
```
Clickstream -> Kafka -> Spark Join (user history + catalog) -> Show recommendations
```
Suggest products at the exact moment of purchase decision.

### 11. Summary and Next Steps

**Kafka Fundamentals (you now understand):**
- Topics organize events by type
- Partitions enable horizontal scaling
- Consumer groups distribute load
- Data is durable and replayable

**PySpark Streaming Pattern:**
1. Create SparkSession with Kafka package
2. `readStream().format('kafka')` to connect
3. Cast value to STRING, parse JSON
4. Transform and flatten fields
5. `writeStream()` to desired sink
6. `.awaitTermination()` to keep running

**Common Issues and Solutions:**

| Issue | Solution |
|---|---|
| No messages in console | Check Terminal 4 sent data. Verify topic exists. |
| Connection refused on :9092 | ZooKeeper or Kafka not running. Check Terminals 1 and 2. |
| JSON parsing error | Schema doesn't match message. Verify Terminal 4 JSON format. |
| Stream blocks forever | Normal. It's waiting for events. Send more messages or Ctrl+C. |
| OutOfMemory error | Reduce batch size or increase heap: `--conf spark.driver.memory=4g` |

**Next: Try these enhancements**
- Add windowed aggregations (count events per 10 seconds)
- Write output to Parquet files
- Add filtering (only temp > 40)
- Connect to external database